# Phase 10 — Testing & Validation (Colab)
Run the complete voice analysis pipeline on 3 test videos using Colab GPU.

## 1. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/voice-analysis-pipeline
!git clone https://github.com/anvay-cpu/voice-analysis-pipeline.git
%cd /content/voice-analysis-pipeline

## 2. Install dependencies

In [ ]:
!pip install -q transformers librosa praat-parselmouth pyyaml psutil noisereduce
!pip install -q --force-reinstall --no-deps speechbrain

## 3. Patch SpeechBrain compatibility
Fixes torchaudio.list_audio_backends and huggingface_hub use_auth_token issues by patching the installed source files directly.

In [ ]:
import subprocess, sys

# Find SpeechBrain install path
sb_path = subprocess.check_output(
    [sys.executable, '-c', 'import importlib; print(importlib.util.find_spec("speechbrain").submodule_search_locations[0])']
).decode().strip()
print(f'SpeechBrain path: {sb_path}')

# --- Patch 1: torchaudio.list_audio_backends ---
backend_file = f'{sb_path}/utils/torch_audio_backend.py'
with open(backend_file, 'r') as f:
    content = f.read()

if 'torchaudio.list_audio_backends()' in content and 'hasattr' not in content:
    content = content.replace(
        'torchaudio.list_audio_backends()',
        '(torchaudio.list_audio_backends() if hasattr(torchaudio, "list_audio_backends") else ["default"])'
    )
    with open(backend_file, 'w') as f:
        f.write(content)
    print('Patch 1 applied: torchaudio.list_audio_backends (inline)')
else:
    print('Patch 1: already applied or not needed')

# --- Patch 2: use_auth_token in fetching.py ---
fetch_file = f'{sb_path}/utils/fetching.py'
with open(fetch_file, 'r') as f:
    content = f.read()

old2 = '"use_auth_token": use_auth_token,'
new2 = '# "use_auth_token": use_auth_token,  # removed for huggingface_hub compat'

if old2 in content:
    content = content.replace(old2, new2)
    with open(fetch_file, 'w') as f:
        f.write(content)
    print('Patch 2 applied: use_auth_token removed')
else:
    print('Patch 2: already applied or not needed')

# --- Patch 3: custom.py 404 in interfaces.py ---
iface_file = f'{sb_path}/inference/interfaces.py'
with open(iface_file, 'r') as f:
    content = f.read()

old3 = '        except ValueError:'
new3 = '        except (ValueError, Exception):'

if old3 in content and new3 not in content:
    content = content.replace(old3, new3, 1)
    with open(iface_file, 'w') as f:
        f.write(content)
    print('Patch 3 applied: custom.py 404 handling')
else:
    print('Patch 3: already applied or not needed')

print('
All patches done. SpeechBrain is ready.')

## 4. Copy large files from Drive

In [ ]:
import os, shutil

os.makedirs('models/disfluency', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/outputs', exist_ok=True)

# Copy disfluency model
src = '/content/drive/MyDrive/best_model.pt'
dst = 'models/disfluency/best_model.pt'
if not os.path.exists(dst):
    print(f'Copying disfluency model ({os.path.getsize(src)/1e6:.0f} MB)...')
    shutil.copy2(src, dst)
    print('Done.')
else:
    print('Disfluency model already in place.')

# Copy test videos
videos = [
    'test_good_speaker_5min.mp4',
    'test_nervous_speaker_5min.mp4',
    'test_monotone_speaker_5min.mp4',
]
for v in videos:
    src = f'/content/drive/MyDrive/test_videos/{v}'
    dst = f'data/raw/{v}'
    if not os.path.exists(dst):
        size = os.path.getsize(src) / 1e6
        print(f'Copying {v} ({size:.1f} MB)...')
        shutil.copy2(src, dst)
    else:
        print(f'{v} already in place.')

print('\nAll files ready!')

## 5. Configure for CUDA & verify models

In [ ]:
import torch, yaml

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Update config for CUDA
with open('configs/pipeline_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
config['pipeline']['device'] = device

with open('configs/pipeline_config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'Config updated to use: {device}')

In [ ]:
# Test loading all models
print('Testing Whisper...')
from src.transcriber import load_whisper
pipe = load_whisper(device=device)
print('  OK')

print('Testing Disfluency...')
from src.disfluency_model import load_disfluency_model
dis_model, dis_proc = load_disfluency_model('models/disfluency/best_model.pt', device=device)
print('  OK')

print('Testing Emotion...')
from src.vocal_emotion import load_emotion_model
emo = load_emotion_model('models/vocal_emotion/best_model.pt', device=device)
print('  OK')

print('\nAll models loaded successfully!')

# Free memory before pipeline run
del pipe, dis_model, dis_proc, emo
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 6. Run Phase 10 — Full Pipeline on 3 Videos

In [ ]:
!python -u run_phase10.py

## 7. Inspect Results

In [ ]:
import json, os

output_dir = 'data/outputs'
for f in sorted(os.listdir(output_dir)):
    if f.endswith('.json'):
        path = os.path.join(output_dir, f)
        with open(path) as fp:
            data = json.load(fp)

        meta = data['metadata']
        summary = data['summary']

        print(f'\n{"="*60}')
        print(f'  {f}')
        print(f'{"="*60}')
        print(f'  Duration:      {meta["duration_sec"]:.0f}s')
        print(f'  Words:         {summary["word_count"]} ({summary["words_per_minute"]:.0f} WPM)')
        print(f'  Fillers:       {summary["filler_count"]} ({summary["fillers_per_minute"]:.1f}/min)')
        print(f'  Disfluencies:  {summary["disfluency_count"]}')
        if summary.get('disfluency_types'):
            print(f'    Types:       {summary["disfluency_types"]}')
        print(f'  Pitch CV:      {summary["avg_pitch_cv"]:.4f}')
        print(f'  Syllable Rate: {summary["avg_syllable_rate"]:.2f} syl/s')
        print(f'  Emotion:       {summary["dominant_emotion"]} (variety: {summary["emotion_variety"]:.2f})')
        if summary.get('emotion_distribution'):
            print(f'    Distribution: {summary["emotion_distribution"]}')
        print(f'  Timings:       {meta["timings"]}')

## 8. Copy outputs back to Drive

In [ ]:
import shutil

drive_output = '/content/drive/MyDrive/voice_pipeline_outputs'
os.makedirs(drive_output, exist_ok=True)

for f in os.listdir('data/outputs'):
    if f.endswith('.json'):
        src = os.path.join('data/outputs', f)
        dst = os.path.join(drive_output, f)
        shutil.copy2(src, dst)
        print(f'Saved: {dst}')

print(f'\nOutputs saved to Drive: {drive_output}')